In [1]:
import pandas as pd
import numpy as np

In [2]:
df_raw = pd.read_csv('messy-food-waste-prediction-dataset/train.csv')
if 'date' in df_raw.columns:
    df_raw['date'] = pd.to_datetime(df_raw['date'])
    df_raw = df_raw.sort_values('date')

In [3]:
split_index = int(len(df_raw) * 0.8)
train_raw = df_raw.iloc[:split_index].copy()
test_raw = df_raw.iloc[split_index:].copy()

print(f"Data dipecah: Train ({len(train_raw)} baris), Test ({len(test_raw)} baris)")

Data dipecah: Train (728 baris), Test (183 baris)


In [4]:
# EKSPERIMEN A: STANDARD PREPROCESSING

train_std = train_raw.copy().drop(columns=['date'], errors='ignore')
test_std = test_raw.copy().drop(columns=['date'], errors='ignore')

# Imputasi Sederhana
num_cols = train_std.select_dtypes(include=['number']).columns.drop('food_waste_kg', errors='ignore')
cat_cols = train_std.select_dtypes(include=['object']).columns

for col in num_cols:
    train_std[col] = train_std[col].fillna(train_std[col].median())
    test_std[col] = test_std[col].fillna(train_std[col].median()) # Pakai median dari train
for col in cat_cols:
    train_std[col] = train_std[col].fillna(train_std[col].mode()[0])
    test_std[col] = test_std[col].fillna(train_std[col].mode()[0])

train_std.to_csv('catboost_train_standard.csv', index=False)
test_std.to_csv('catboost_test_standard.csv', index=False)

C:\Users\davew\AppData\Local\Temp\ipykernel_16740\3717779426.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = train_std.select_dtypes(include=['object']).columns


In [5]:
# EKSPERIMEN B: ROBUST PREPROCESSING

train_rob = train_raw.copy().drop_duplicates()
test_rob = test_raw.copy()

# 1. Pembersihan Teks
for col in cat_cols:
    train_rob[col] = train_rob[col].astype(str).str.lower().str.strip().str.replace(r'[^\w\s]', '', regex=True)
    test_rob[col] = test_rob[col].astype(str).str.lower().str.strip().str.replace(r'[^\w\s]', '', regex=True)

# 2. Imputasi
for col in num_cols:
    train_rob[col] = train_rob[col].fillna(train_rob[col].median())
    test_rob[col] = test_rob[col].fillna(train_rob[col].median())
for col in cat_cols:
    train_rob[col] = train_rob[col].replace('nan', np.nan).fillna(train_rob[col].mode()[0])
    test_rob[col] = test_rob[col].replace('nan', np.nan).fillna(train_rob[col].mode()[0])
    
# 3. Penanganan Outlier (HANYA pada Train data, Test data dibiarkan apa adanya)
for col in num_cols:
    Q1 = train_rob[col].quantile(0.25)
    Q3 = train_rob[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    train_rob[col] = np.clip(train_rob[col], lower, upper)

# 4. Feature Engineering (Lag & Tanggal)
if 'date' in train_raw.columns:
    for df_temp, df_orig in zip([train_rob, test_rob], [train_raw, test_raw]):
        df_temp['hari_dalam_minggu'] = df_orig['date'].dt.dayofweek
        df_temp['is_weekend'] = df_temp['hari_dalam_minggu'].apply(lambda x: 1 if x >= 5 else 0)
        df_temp['limbah_h_minus_1'] = df_orig['food_waste_kg'].shift(1).fillna(df_orig['food_waste_kg'].median())
        df_temp.drop(columns=['date'], inplace=True, errors='ignore')

# Ekspor Data Robust
train_rob.to_csv('catboost_train_robust.csv', index=False)
test_rob.to_csv('catboost_test_robust.csv', index=False)

print("Preprocessing Selesai. Data Standard dan Robust berhasil diekspor!")

Preprocessing Selesai. Data Standard dan Robust berhasil diekspor!
